# AEM4L5 · Notebook 2 — Deploy: serverless vs servidor

**Clase:** Arquitecturas avanzadas de adaptación.

Cómo decidir un patrón de despliegue para sistemas de IA.

## 0. Setup tecnico

Instalamos solo `langchain-core` (no requiere API key). `langchain-openai` es **opcional**.

Toda la clase corre **sin claves**: simulamos el modelo con `RunnableLambda`.

In [ ]:
# Dependencia base (no requiere API key):
!pip install -q langchain-core

# Opcional, solo si quisieras usar OpenAI real (no es necesario para esta clase):
# !pip install -q langchain-openai

In [ ]:
import time
import asyncio
import cProfile
import pstats
import io
import re
from typing import List, Dict, Any

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel


def fake_llm_response(prompt_value):
    # Simulador de modelo: evita depender de APIs pagas.
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    return f"Respuesta simulada del modelo para:\n{text[:300]}"


fake_llm = RunnableLambda(fake_llm_response)
print("Setup listo. Modelo simulado disponible como fake_llm.")

## 1. Objetivo del notebook

Que puedas decidir un **patrón de despliegue** y explicar: deploy, serving, endpoint, serverless, servidor persistente, cold start, latencia, throughput, costo fijo y variable, y **qué patrón conviene** según tráfico y restricciones.

## 2. Mapa visual del tema

```mermaid
flowchart TB
    A[Solicitud de usuario] --> B{Tipo de despliegue}
    B --> C[Serverless]
    B --> D[Servidor persistente]
    C --> C1[Escala a cero]
    C --> C2[Paga por uso]
    C --> C3[Puede tener cold start]
    D --> D1[Modelo siempre cargado]
    D --> D2[Baja latencia inicial]
    D --> D3[Costo fijo]
    C3 --> E[Bueno para trafico irregular]
    D2 --> F[Bueno para trafico constante]
```

## 3. Glosario

| Término | Explicación |
|---|---|
| Deploy | Publicar una aplicación o modelo para que pueda ser usado. |
| Serving | Servir inferencias mediante una API o endpoint. |
| Endpoint | URL donde una aplicación recibe solicitudes. |
| Serverless | Ejecución bajo demanda con infraestructura gestionada. |
| Servidor persistente | Máquina o contenedor siempre encendido. |
| Cold start | Demora inicial cuando una función arranca desde cero. |
| Latencia | Tiempo total hasta recibir una respuesta. |
| Throughput | Cantidad de solicitudes procesadas por unidad de tiempo. |
| Escalabilidad | Capacidad de soportar más carga. |
| Costo fijo | Costo aunque no haya usuarios. |
| Costo variable | Costo proporcional al uso. |
| SLA | Compromiso de disponibilidad o rendimiento. |
| Infraestructura ociosa | Recursos encendidos pero sin uso. |

## 4. Teoría explicada paso a paso

### 4.1. Basado en servidor (persistente)

Un servidor persistente mantiene la aplicación y el modelo cargados.

- **Ventajas:** menor latencia inicial; modelo ya cargado; más control; bueno para tráfico constante y experiencias en tiempo real.
- **Desventajas:** costo aunque no haya tráfico; más mantenimiento; requiere monitoreo y escalado planificado.

### 4.2. Serverless

La plataforma ejecuta funciones o contenedores **bajo demanda**.

- **Ventajas:** pago por uso; escala automática; menos carga operativa; bueno para tráfico irregular.
- **Desventajas:** cold start; límites de memoria y tiempo; menos control; latencia más variable; puede encarecerse con tráfico alto y sostenido.

## 5. Tabla comparativa

| Criterio | Serverless | Servidor persistente |
|---|---|---|
| Costo sin tráfico | Bajo | Alto |
| Latencia inicial | Puede ser alta | Baja |
| Control técnico | Menor | Mayor |
| Operación | Más simple | Más compleja |
| Escalado | Automático | Configurado por el equipo |
| Ideal para | Uso irregular | Uso constante |
| Riesgo principal | Cold start | Costo ocioso |
| Modelo grande | Puede ser difícil | Más manejable |
| Tiempo real | Riesgoso | Mejor opción |

## 6. Gráfico Mermaid: LangChain dentro de una API

```mermaid
flowchart LR
    A[Cliente] --> B[API Endpoint]
    B --> C[LangChain Chain]
    C --> D[Prompt]
    D --> E[Modelo o endpoint adaptado]
    E --> F[Respuesta]
    F --> A
```

## 7. Mini ejemplo conceptual

La decisión de despliegue se puede expresar como **reglas**: dado el tráfico, la latencia exigida, el presupuesto y el tamaño del modelo, qué patrón conviene.

In [ ]:
def elegir_arquitectura(trafico, latencia_estricta, presupuesto_bajo, modelo_grande):
    if latencia_estricta and trafico == "constante":
        return "Servidor persistente"
    if modelo_grande and latencia_estricta:
        return "Servidor persistente"
    if presupuesto_bajo and trafico == "irregular":
        return "Serverless"
    if trafico == "picos":
        return "Serverless o hibrido"
    return "Depende: analizar costo, latencia, operacion y crecimiento"

## 8. Código Python + LangChain

Probamos la función de decisión con casos reales.

In [ ]:
casos = [
    {"nombre": "App interna usada 10 veces por dia", "trafico": "irregular", "latencia_estricta": False, "presupuesto_bajo": True, "modelo_grande": False},
    {"nombre": "Asistente de voz en tiempo real", "trafico": "constante", "latencia_estricta": True, "presupuesto_bajo": False, "modelo_grande": True},
    {"nombre": "Analisis legal con picos mensuales", "trafico": "picos", "latencia_estricta": False, "presupuesto_bajo": True, "modelo_grande": False},
    {"nombre": "API usada 24/7", "trafico": "constante", "latencia_estricta": False, "presupuesto_bajo": False, "modelo_grande": False},
]

for caso in casos:
    decision = elegir_arquitectura(
        trafico=caso["trafico"],
        latencia_estricta=caso["latencia_estricta"],
        presupuesto_bajo=caso["presupuesto_bajo"],
        modelo_grande=caso["modelo_grande"],
    )
    print(caso["nombre"], "->", decision)

Una chain de LangChain puede vivir dentro de un endpoint FastAPI, una función serverless, un worker o un backend persistente. La chain es la **misma**; lo que cambia es **dónde** se ejecuta.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

prompt = ChatPromptTemplate.from_template(
    "Sos un asistente de resumenes. Resumi este documento en {cantidad_lineas} lineas:\n\n{documento}"
)

def fake_model(prompt_value):
    text = prompt_value.to_string()
    return f"Resumen simulado generado por el modelo:\n{text[:250]}..."

summary_chain = prompt | RunnableLambda(fake_model)

print(summary_chain.invoke({
    "cantidad_lineas": 5,
    "documento": "Documento largo de ejemplo..."
}))

## 9. Ejercicio guiado — Elegir arquitectura

Completá la arquitectura recomendada para cada caso (y el motivo).

| Caso | Tráfico | Latencia | Presupuesto |
|---|---|---|---|
| App interna de bajo uso | Irregular | Flexible | Bajo |
| Asistente de voz | Constante | Estricta | Medio/alto |
| Procesamiento mensual de documentos | Picos | Flexible | Bajo |
| API 24/7 | Constante | Media | Medio |

In [ ]:
solucion = {
    "App interna de bajo uso": ("Serverless", "Evita pagar infraestructura ociosa."),
    "Asistente de voz": ("Servidor persistente", "Necesita baja latencia y modelo caliente."),
    "Procesamiento mensual": ("Serverless o hibrido", "Trafico por picos."),
    "API 24/7": ("Servidor persistente", "Trafico sostenido."),
}
for caso, (arq, motivo) in solucion.items():
    print(f"- {caso}: {arq}  ({motivo})")

## 10. Preguntas de interpretación (matriz de trade-offs)

1. ¿Qué duele más: costo fijo o cold start?
2. ¿La experiencia de usuario tolera demora inicial?
3. ¿El modelo es grande?
4. ¿El tráfico es predecible?
5. ¿Cuándo migrarías de serverless a servidor?

## 11. Errores comunes

- Creer que serverless siempre es más barato.
- Ignorar cold starts.
- Usar servidor persistente para tráfico casi nulo.
- No medir latencia total.
- Confundir latencia de inferencia con latencia de pipeline.
- No planificar migración si el tráfico cambia.

## 12. Cierre conceptual

No hay un patrón “mejor”: hay un patrón **adecuado al tráfico, la latencia y el presupuesto**. Serverless gana con uso irregular y presupuesto bajo; servidor persistente gana con tráfico constante y latencia estricta. Lo profesional es **medir** y prever la **migración** cuando el tráfico cambia.